# Basic RAG — Chat with Your Documents

Build a Retrieval-Augmented Generation pipeline that answers questions grounded in your own documents.

**Topics:** Document loading, text splitting, embeddings, ChromaDB vector store, similarity search, QA pipeline

## 1. Loading and Preprocessing Documents

In [ ]:
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Document:
    """A loaded document with text and metadata."""
    content: str
    source: str
    metadata: dict = field(default_factory=dict)
    
    @property
    def word_count(self) -> int:
        return len(self.content.split())

def load_text_file(path: str) -> Document:
    p = Path(path)
    return Document(content=p.read_text(encoding='utf-8'), source=str(p))

def load_markdown(path: str) -> Document:
    p = Path(path)
    raw = p.read_text(encoding='utf-8')
    # Strip markdown syntax for cleaner embedding
    clean = re.sub(r'#{1,6}\s', '', raw)     # headers
    clean = re.sub(r'`{1,3}[^`]*`{1,3}', '', clean)  # code
    clean = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', clean)  # links
    return Document(content=clean.strip(), source=str(p), metadata={'type': 'markdown'})

def load_from_url(url: str) -> Document:
    """Load text content from a URL."""
    import urllib.request
    from html.parser import HTMLParser
    
    class TextExtractor(HTMLParser):
        def __init__(self):
            super().__init__()
            self.text = []
        def handle_data(self, data):
            self.text.append(data)
    
    with urllib.request.urlopen(url) as r:
        html = r.read().decode('utf-8')
    parser = TextExtractor()
    parser.feed(html)
    return Document(content=' '.join(parser.text), source=url)

# Demo with in-memory content
sample_doc = Document(
    content='Machine learning is a subset of AI. Neural networks learn from data through backpropagation.',
    source='ml_intro.txt',
    metadata={'topic': 'ML', 'level': 'beginner'},
)
print(f'Document: {sample_doc.source}')
print(f'Words: {sample_doc.word_count}')
print(f'Preview: {sample_doc.content[:80]}...')

## 2. Text Splitting Strategies

In [ ]:
from dataclasses import dataclass

@dataclass
class Chunk:
    text: str
    source: str
    chunk_index: int
    char_start: int
    char_end: int

def split_by_tokens(
    doc: Document,
    chunk_size: int = 500,
    overlap: int = 50,
) -> list[Chunk]:
    """Split on word boundaries with overlap for context continuity."""
    words = doc.content.split()
    chunks = []
    i = 0
    idx = 0
    while i < len(words):
        chunk_words = words[i:i + chunk_size]
        text = ' '.join(chunk_words)
        start_char = len(' '.join(words[:i]))
        chunks.append(Chunk(
            text=text,
            source=doc.source,
            chunk_index=idx,
            char_start=start_char,
            char_end=start_char + len(text),
        ))
        i += chunk_size - overlap  # slide forward with overlap
        idx += 1
    return chunks

def split_by_sentences(doc: Document, max_sentences: int = 5) -> list[Chunk]:
    """Split on sentence boundaries — more semantically meaningful."""
    sentences = re.split(r'(?<=[.!?])\s+', doc.content)
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        group = sentences[i:i + max_sentences]
        text = ' '.join(group)
        chunks.append(Chunk(text=text, source=doc.source, chunk_index=i//max_sentences, char_start=0, char_end=len(text)))
    return chunks

# Long sample document
long_text = ' '.join([
    'Transformers revolutionized NLP in 2017.',
    'The attention mechanism allows models to focus on relevant parts of the input.',
    'BERT uses bidirectional attention to understand context from both directions.',
    'GPT models use causal (unidirectional) attention for generation tasks.',
    'RAG combines retrieval with generation to ground answers in documents.',
] * 10)

doc = Document(content=long_text, source='demo.txt')
chunks = split_by_tokens(doc, chunk_size=50, overlap=10)

print(f'Document: {doc.word_count} words')
print(f'Chunks (size=50, overlap=10): {len(chunks)}')
print(f'\nFirst chunk ({len(chunks[0].text.split())} words):')
print(f'  "{chunks[0].text[:100]}..."')
print(f'\nChunk overlap demo — end of chunk 0:')
print(f'  "{" ".join(chunks[0].text.split()[-10:])}"')
print(f'Start of chunk 1:')
print(f'  "{" ".join(chunks[1].text.split()[:10])}"')

## 3. Creating Embeddings

In [ ]:
import numpy as np
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))

def embed_openai(texts: list[str], model: str = 'text-embedding-3-small') -> np.ndarray:
    """Get embeddings from OpenAI — fast, cheap, high quality."""
    response = client.embeddings.create(model=model, input=texts)
    return np.array([item.embedding for item in response.data])

def embed_local(texts: list[str], model_name: str = 'all-MiniLM-L6-v2') -> np.ndarray:
    """Local embeddings using sentence-transformers — free, private."""
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name)
    return model.encode(texts, convert_to_numpy=True)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Measure similarity between two vectors (1.0 = identical, 0.0 = unrelated)."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Model comparison
models = [
    ('text-embedding-3-small', 1536, '$0.02/1M tokens', 'OpenAI — best for production'),
    ('text-embedding-3-large', 3072, '$0.13/1M tokens', 'OpenAI — highest accuracy'),
    ('all-MiniLM-L6-v2', 384, 'Free', 'Local — fast, good for experiments'),
    ('bge-large-en-v1.5', 1024, 'Free', 'Local — near OpenAI quality'),
]
print(f'{"Model":<30} {"Dims":<6} {"Cost":<20} {"Notes"}')
print('-' * 75)
for name, dims, cost, notes in models:
    print(f'{name:<30} {dims:<6} {cost:<20} {notes}')
print()

# Simulate similarity scores
queries = ['What is a neural network?', 'How do I cook pasta?']
doc_text = 'Neural networks are computing systems inspired by the human brain.'
print('Simulated similarity scores:')
for q, score in zip(queries, [0.87, 0.12]):
    match = '✓ RETRIEVED' if score > 0.7 else '✗ skipped'
    print(f'  Q: "{q}" → sim={score:.2f} {match}')

## 4. Vector Store with ChromaDB

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

def build_vector_store(
    chunks: list[Chunk],
    collection_name: str = 'knowledge_base',
    persist_dir: str = './chroma_db',
) -> chromadb.Collection:
    """Build a persistent ChromaDB vector store from document chunks."""
    client_chroma = chromadb.PersistentClient(path=persist_dir)
    
    # Use OpenAI embeddings via ChromaDB's built-in function
    ef = embedding_functions.OpenAIEmbeddingFunction(
        api_key=os.getenv('OPENAI_API_KEY'),
        model_name='text-embedding-3-small',
    )
    
    collection = client_chroma.get_or_create_collection(
        name=collection_name,
        embedding_function=ef,
        metadata={'hnsw:space': 'cosine'},  # cosine similarity
    )
    
    # Upsert chunks (safe to re-run)
    collection.upsert(
        ids=[f'{c.source}_{c.chunk_index}' for c in chunks],
        documents=[c.text for c in chunks],
        metadatas=[{'source': c.source, 'chunk_index': c.chunk_index} for c in chunks],
    )
    return collection

def search(collection: chromadb.Collection, query: str, n_results: int = 3) -> list[dict]:
    """Retrieve the most relevant chunks for a query."""
    results = collection.query(query_texts=[query], n_results=n_results)
    return [
        {'text': doc, 'source': meta['source'], 'distance': dist}
        for doc, meta, dist in zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0],
        )
    ]

# Simulated search results
simulated_results = [
    {'text': 'Transformers use self-attention to process sequences in parallel.', 'source': 'dl_book.txt', 'distance': 0.12},
    {'text': 'The attention mechanism assigns weights to relevant input tokens.', 'source': 'papers.txt', 'distance': 0.18},
    {'text': 'BERT bidirectional context understanding revolutionized NLP.', 'source': 'blog.txt', 'distance': 0.25},
]
print('Simulated search results for "How does attention work?":')
for i, r in enumerate(simulated_results, 1):
    print(f'  [{i}] (dist={r["distance"]:.2f}) {r["text"][:70]}...')
    print(f'       Source: {r["source"]}')

## 5. End-to-End RAG Pipeline

In [ ]:
from openai import OpenAI

RAG_SYSTEM = """You are a helpful assistant. Answer questions using ONLY the provided context.
If the answer is not in the context, say: "I don't have enough information to answer this."
Always cite your sources using [1], [2] etc."""

RAG_USER_TEMPLATE = """Context:
{context}

Question: {question}

Answer:"""

class RAGPipeline:
    """End-to-end RAG: retrieve relevant chunks → augment prompt → generate answer."""
    
    def __init__(self, collection, llm_model: str = 'gpt-4o-mini', top_k: int = 3):
        self.collection = collection
        self.llm = OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))
        self.llm_model = llm_model
        self.top_k = top_k
    
    def answer(self, question: str) -> dict:
        # Step 1: Retrieve
        retrieved = search(self.collection, question, n_results=self.top_k)
        
        # Step 2: Augment
        context = '\n\n'.join(
            f'[{i+1}] {r["text"]} (Source: {r["source"]})'
            for i, r in enumerate(retrieved)
        )
        prompt = RAG_USER_TEMPLATE.format(context=context, question=question)
        
        # Step 3: Generate
        response = self.llm.chat.completions.create(
            model=self.llm_model,
            messages=[
                {'role': 'system', 'content': RAG_SYSTEM},
                {'role': 'user', 'content': prompt},
            ],
        )
        
        return {
            'question': question,
            'answer': response.choices[0].message.content,
            'sources': [r['source'] for r in retrieved],
            'n_retrieved': len(retrieved),
        }

# RAG pipeline flow visualization
print('RAG Pipeline — 3 steps:')
print()
print('  User Question')
print('       │')
print('       ▼')
print('  [1] RETRIEVE — embed question → search vector store → top-k chunks')
print('       │')
print('       ▼')
print('  [2] AUGMENT  — inject chunks into prompt as context')
print('       │')
print('       ▼')
print('  [3] GENERATE — LLM reads context + question → grounded answer')
print()
print('Key benefit: LLM cannot hallucinate facts not in the retrieved context.')

## Exercises

1. **Multi-file RAG:** Extend the pipeline to load all `.txt` and `.md` files from a directory, chunk them, index into ChromaDB, and expose a CLI interface that accepts a question and returns an answer with citations.

2. **Retrieval evaluation:** Implement a function that tests retrieval quality by taking 5 question-answer pairs, running retrieval, and checking whether the correct source document appears in the top-k results. Report recall@3 and recall@5.

3. **Metadata filtering:** Add support for filtering by document type (e.g., only search in `.pdf` sources or only in chunks tagged with `topic='ML'`). Implement this using ChromaDB's `where` parameter.